# Apply a business rule to a table

## Goal

A data pipeline often needs a business decision for every record: route an order, assign a category, or check eligibility. Rulebricks lets the people responsible for that policy maintain it as a versioned rule. Spark supplies the records and keeps the decisions with the source data.

Here, a rule decides which orders need review. You will send a small table in bulk, save the results, and produce a table ready for downstream use. The same operation accepts a large source table.

| Starting point | Rulebricks adds | Finished dataset |
| --- | --- | --- |
| One row per order | A decision from the selected rule version | Original orders, typed decisions, and execution details |

A **rule** is one decision table or set of conditions. The next notebook introduces a **flow**, which combines decision steps.

## Setup

Use a Spark 3.5 session with Delta Lake and `rulebricks-spark` installed. Follow the [installation guide](../docs/getting-started.md) for credentials and storage.

Edit the API URL, rule identifier, version, and output location in the cell below. The local output path works for a local Spark session; choose shared storage on a cluster. The API key comes from `RULEBRICKS_API_KEY` or your secret manager.

This walkthrough expects the following example rule, published in Rulebricks:

| Inputs | Policy | Outputs |
| --- | --- | --- |
| `request_id`: string, `amount`: number, `country`: string | Approve amounts below 100 in US/CA; review everything else | `decision`: string, `score`: number equal to the amount |

The API key needs permission to execute the rule and read its published schema.

In [ ]:
import os
import uuid

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType

from rulebricks_spark import apply_rule

spark = SparkSession.getActiveSession() or SparkSession.builder.appName("bulk-rules").getOrCreate()
api_key = os.environ["RULEBRICKS_API_KEY"]
base_url = "https://rulebricks.acme.com/api/v1"
rule_slug = "order-routing"
rule_version = "1"
output_root = "file:///tmp/rulebricks-examples"
run_id = str(uuid.uuid4())
result_path = f"{output_root}/{run_id}/decisions"

## 1. Choose the records

These four orders cover both outcomes. `request_id` lets you match a decision to its original record. For your own pipeline, replace this cell with a table read and use your existing record ID.

In [ ]:
source = spark.createDataFrame(
    [
        (f"{run_id}-1", 50.0, "US"),
        (f"{run_id}-2", 150.0, "CA"),
        (f"{run_id}-3", 20.0, "GB"),
        (f"{run_id}-4", 75.0, "CA"),
    ],
    "request_id string, amount double, country string",
)
source.show(truncate=False)

## 2. Apply the rule and save the results

`apply_rule()` groups records into requests and keeps the responses aligned with their inputs. The selected rule version supplies the output column types. `input_validation="strict"` also checks that the source has the expected input columns and compatible types before execution.

| Column after execution | What it contains |
| --- | --- |
| `request_id`, `amount`, `country` | Your original values |
| `rulebricks_result` | A struct with `decision` and `score` |
| `rulebricks_meta` | Success or failure, version, timing, and available execution IDs |

Spark runs the calls when the result is written. Reading the saved table afterward lets us inspect it without executing the rule again.
The rule computes `score` with a function column, so its runtime type is declared explicitly as a Spark `double` below.


In [ ]:
evaluated = apply_rule(
    source,
    rule_slug,
    version=rule_version,
    output_schema=StructType().add("decision", "string").add("score", "double"),
    api_key=api_key,
    base_url=base_url,
    correlation_id_path="request_id",
    input_validation="strict",
)
evaluated.write.format("delta").mode("errorifexists").save(result_path)
saved = spark.read.format("delta").load(result_path)

## 3. Check the run and use the decisions

Every input should have one outcome. A failure stays attached to its source row, with an explanation in `rulebricks_meta.error`. The successful decisions below become ordinary Spark columns.

In [ ]:
assert saved.count() == 4
saved.groupBy("rulebricks_meta.status").count().show()
assert saved.filter("rulebricks_meta.status != 'ok'").count() == 0

decisions = saved.select(
    "request_id",
    "amount",
    "country",
    "rulebricks_result.decision",
    "rulebricks_result.score",
)
decisions.show(truncate=False)
assert decisions.filter("decision = 'approve'").count() == 2
assert decisions.filter("decision = 'review'").count() == 2

## Next steps

Keep the saved dataset as the run record. The request ID connects each outcome to your data; it does not prevent duplicate execution if a job is rerun.

Continue with [bulk flows](02-bulk-flows.ipynb) for multi-step decisions and the main throughput controls. For custom input checks or changing output shapes, see [working with results](../docs/working-with-results.md).